# Notebook 5: Model Selection (Improved)

This notebook upgrades model selection from pure forecasting to forecasting + driver decision intelligence.

## What this notebook does
- Time-aware model selection with Optuna.
- Compare models (`LR`, `Ridge`, `RF`, `GBR`, `XGBR` when available).
- Evaluate with `MAPE`, `MAE`, `RMSE`, `sMAPE`, `R2`.
- Generate business outputs from predictions:
  - surge detection
  - risk/stability score
  - revenue estimation
  - demand pressure
  - relocation recommendation
  - best time recommendation


In [ ]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score,
)

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    import mlflow
    HAS_MLFLOW = True
except Exception:
    HAS_MLFLOW = False

try:
    import dagshub
    HAS_DAGSHUB = True
except Exception:
    HAS_DAGSHUB = False

import optuna

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

data_interim = project_root / "data" / "interim"
data_processed = project_root / "data" / "processed"
models_dir = project_root / "models"
reports_dir = project_root / "reports"

for folder in [data_interim, data_processed, models_dir, reports_dir]:
    folder.mkdir(parents=True, exist_ok=True)

# Tracking flags (safe defaults for Kaggle/local)
USE_MLFLOW = False
USE_DAGSHUB = False

if USE_MLFLOW and HAS_MLFLOW:
    mlflow.set_experiment("Model Selection")
    if USE_DAGSHUB and HAS_DAGSHUB:
        # Uncomment and update if you want DagsHub tracking
        # dagshub.init(repo_owner="<owner>", repo_name="<repo>", mlflow=True)
        pass


In [ ]:
def smape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.abs(y_true) + np.abs(y_pred)
    return np.mean(2.0 * np.abs(y_true - y_pred) / np.where(denom == 0, 1.0, denom))


def safe_mape(y_true, y_pred, eps: float = 1.0):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs(y_true - y_pred) / np.maximum(np.abs(y_true), eps))


def evaluate_metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE": float(safe_mape(y_true, y_pred)),
        "sMAPE": float(smape(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }


def safe_fill_frame(df: pd.DataFrame):
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_numeric_dtype(out[col]):
            median_val = out[col].median()
            if pd.isna(median_val):
                median_val = 0.0
            out[col] = out[col].fillna(median_val)
        else:
            mode_vals = out[col].mode(dropna=True)
            fill_val = mode_vals.iloc[0] if len(mode_vals) else "unknown"
            out[col] = out[col].fillna(fill_val)
    return out


def get_time_col(df: pd.DataFrame):
    for c in ["pickup_slot", "tpep_pickup_datetime", "timestamp"]:
        if c in df.columns:
            return c
    return None


In [ ]:
# Load data for model selection (prefer fresh historical split to avoid stale leakage)
train_path = data_processed / "train.csv"
test_path = data_processed / "test.csv"

local_hist_candidates = [
    data_interim / "historical_features.csv",
    data_interim / "final_data.csv",
]
kaggle_hist_candidates = [
    Path("/kaggle/working/historical_features.csv"),
    Path("/kaggle/working/final_data.csv"),
    Path("/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/historical_features.csv"),
    Path("/kaggle/input/notebooks/shubhamyadav74/notebookacca71d522/final_data.csv"),
]
hist_candidates = local_hist_candidates + kaggle_hist_candidates

FORCE_REBUILD_SPLIT = True

hist_path = next((p for p in hist_candidates if p.exists()), None)

if FORCE_REBUILD_SPLIT and hist_path is not None:
    all_df = pd.read_csv(hist_path)
    time_col = get_time_col(all_df)
    if time_col is None:
        raise ValueError("Historical data missing time column (`pickup_slot` or `tpep_pickup_datetime`).")

    all_df[time_col] = pd.to_datetime(all_df[time_col], errors="coerce")
    all_df = all_df.dropna(subset=[time_col]).copy()

    if "region" in all_df.columns:
        all_df["region"] = pd.to_numeric(all_df["region"], errors="coerce")
        all_df = all_df.dropna(subset=["region"]).copy()
        all_df["region"] = all_df["region"].astype(int)
        all_df = all_df.sort_values([time_col, "region"]).reset_index(drop=True)
    else:
        all_df = all_df.sort_values(time_col).reset_index(drop=True)

    unique_times = np.sort(all_df[time_col].unique())
    if len(unique_times) < 2:
        raise ValueError("Not enough unique timestamps to create a chronological train/test split.")

    cut_idx = max(1, int(len(unique_times) * 0.8))
    cut_idx = min(cut_idx, len(unique_times) - 1)
    cutoff_time = unique_times[cut_idx - 1]

    train_df = all_df[all_df[time_col] <= cutoff_time].copy()
    test_df = all_df[all_df[time_col] > cutoff_time].copy()

    source_mode = f"historical_time_split ({Path(hist_path).name})"

    # Save split for reproducibility in current run environment.
    try:
        train_df.to_csv(train_path, index=False)
        test_df.to_csv(test_path, index=False)
    except Exception:
        pass

elif train_path.exists() and test_path.exists():
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    source_mode = "processed_train_test"
else:
    raise FileNotFoundError(
        "No historical features found and no train/test split available. Run Notebook 4 first."
    )

print("Source mode:", source_mode)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

_load_time_col = get_time_col(train_df)
if _load_time_col is not None and _load_time_col in test_df.columns:
    print("Train range:", pd.to_datetime(train_df[_load_time_col]).min(), "->", pd.to_datetime(train_df[_load_time_col]).max())
    print("Test range :", pd.to_datetime(test_df[_load_time_col]).min(), "->", pd.to_datetime(test_df[_load_time_col]).max())



In [ ]:
# Target and feature preparation + strict no-leak lag engineering
# IMPORTANT: model-selection must use raw demand target, not smoothed/model target.
LEAKAGE_GUARD_VERSION = "v2_2026-04-14"
print("Leakage guard:", LEAKAGE_GUARD_VERSION)

target_candidates = ["total_pickups_raw", "total_pickups"]
target_col = next((c for c in target_candidates if c in train_df.columns), None)

if target_col is None:
    if "total_pickups_model" in train_df.columns:
        raise ValueError(
            "Leakage-safe target not found. `total_pickups_model` is smoothed and should not be used for model selection. "
            "Use Notebook 4 output that includes `total_pickups_raw` (or `total_pickups`)."
        )
    raise ValueError(f"Target column missing. Expected one of: {target_candidates}")

time_col = get_time_col(train_df)
if time_col is None:
    raise ValueError("Time column is required for lag feature creation (`pickup_slot` or `tpep_pickup_datetime`).")

# Keep these only for post-prediction business outputs (not model fitting)
context_keep_cols = [
    c
    for c in [
        time_col,
        "region",
        "rolling_mean",
        "rolling_std",
        "avg_fare_region_slot",
        "avg_pickups",
        "avg_pickups_ewm",
    ]
    if c in test_df.columns
]

# 15-min lag setup
lag_steps = [1, 2, 3, 6, 12, 96]
rolling_windows = [3, 6]


def add_lag_columns(df: pd.DataFrame, target: str, t_col: str):
    out = df.copy()
    out[t_col] = pd.to_datetime(out[t_col], errors="coerce")
    out = out.dropna(subset=[t_col]).copy()

    if "region" in out.columns:
        out["region"] = pd.to_numeric(out["region"], errors="coerce")
        out = out.dropna(subset=["region"]).copy()
        out["region"] = out["region"].astype(int)
        out = out.sort_values(["region", t_col]).reset_index(drop=True)

        for lag in lag_steps:
            out[f"lag_{lag}"] = out.groupby("region")[target].shift(lag)

        for w in rolling_windows:
            out[f"lag_roll_mean_{w}"] = out.groupby("region")[target].transform(
                lambda s: s.shift(1).rolling(window=w, min_periods=w).mean()
            )
            out[f"lag_roll_std_{w}"] = out.groupby("region")[target].transform(
                lambda s: s.shift(1).rolling(window=w, min_periods=w).std()
            )
    else:
        out = out.sort_values([t_col]).reset_index(drop=True)

        for lag in lag_steps:
            out[f"lag_{lag}"] = out[target].shift(lag)

        for w in rolling_windows:
            out[f"lag_roll_mean_{w}"] = out[target].shift(1).rolling(window=w, min_periods=w).mean()
            out[f"lag_roll_std_{w}"] = out[target].shift(1).rolling(window=w, min_periods=w).std()

    lag_cols = [c for c in out.columns if c.startswith("lag_")]
    return out, lag_cols


# Keep raw copies for leakage-safe test lag creation.
train_source = train_df.copy()
test_source = test_df.copy()

# Train lag features
train_with_lags, lag_cols_train = add_lag_columns(train_source, target_col, time_col)
before_train = len(train_with_lags)
train_df = train_with_lags.dropna(subset=lag_cols_train).reset_index(drop=True)
dropped_train = before_train - len(train_df)

# Test lag features using trailing TRAIN history (region-wise)
max_hist = max(lag_steps + rolling_windows) + 2
if "region" in train_source.columns:
    train_hist_base = train_source.copy()
    train_hist_base[time_col] = pd.to_datetime(train_hist_base[time_col], errors="coerce")
    train_hist_base["region"] = pd.to_numeric(train_hist_base["region"], errors="coerce")
    train_hist_base = train_hist_base.dropna(subset=[time_col, "region"]).copy()
    train_hist_base["region"] = train_hist_base["region"].astype(int)
    train_hist_base = train_hist_base.sort_values(["region", time_col]).reset_index(drop=True)
    history = train_hist_base.groupby("region", group_keys=False).tail(max_hist)
else:
    train_hist_base = train_source.copy()
    train_hist_base[time_col] = pd.to_datetime(train_hist_base[time_col], errors="coerce")
    train_hist_base = train_hist_base.dropna(subset=[time_col]).sort_values(time_col).reset_index(drop=True)
    history = train_hist_base.tail(max_hist)

history = history.copy()
history["_is_test"] = 0

test_tagged = test_source.copy()
test_tagged["_is_test"] = 1

test_temp = pd.concat([history, test_tagged], axis=0, ignore_index=True)
test_temp_with_lags, lag_cols_test = add_lag_columns(test_temp, target_col, time_col)

test_df = test_temp_with_lags[test_temp_with_lags["_is_test"] == 1].copy()
test_df = test_df.drop(columns=["_is_test"], errors="ignore")

lag_feature_cols = sorted(list(set(lag_cols_train).intersection(set(lag_cols_test))))
if not lag_feature_cols:
    raise ValueError("No common lag features found between train and test after lag processing.")

before_test = len(test_df)
test_df = test_df.dropna(subset=lag_feature_cols).reset_index(drop=True)
dropped_test = before_test - len(test_df)

# Strict whitelist only
allowed_time_cols = [
    "region",
    "pickup_day_of_week",
    "pickup_hour",
    "day_of_week",
    "month",
    "is_weekend",
    "rush_hour",
    "is_night",
]

feature_cols = [c for c in allowed_time_cols if c in train_df.columns and c in test_df.columns]
feature_cols += [c for c in lag_feature_cols if c in train_df.columns and c in test_df.columns]

if not feature_cols:
    raise ValueError("No features available after strict allowlist filtering.")

# Hard safety: ensure forbidden columns never enter model
forbidden_prefixes = ("predicted_", "rolling_", "surge_", "risk_", "expected_")
forbidden_exact = {
    "total_revenue", "total_tip", "total_trip_distance", "avg_pickups", "avg_pickups_ewm",
    "avg_pickups_ewm_tuned", "avg_pickups_ma_tuned", "fare_per_km", "tip_ratio", "tip_per_pickup",
    "revenue_density_15min", "avg_fare_region_slot", "avg_speed_kmh", "demand_pressure",
    "total_pickups_model", target_col,
}

bad_features = [
    c for c in feature_cols
    if c in forbidden_exact or any(c.lower().startswith(p) for p in forbidden_prefixes)
]
if bad_features:
    raise ValueError(f"Leakage detected in feature list: {bad_features}")

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

context_test = test_df[context_keep_cols].copy() if context_keep_cols else pd.DataFrame(index=test_df.index)

X_train = safe_fill_frame(X_train)
X_test = safe_fill_frame(X_test)

# Additional hard leakage check: no feature should be identical to target.
exact_match_cols = []
for c in X_train.columns:
    if pd.api.types.is_numeric_dtype(X_train[c]):
        match_ratio = np.mean(np.isclose(X_train[c].to_numpy(), y_train.to_numpy(), rtol=0, atol=1e-12))
        if match_ratio > 0.999:
            exact_match_cols.append((c, float(match_ratio)))

if exact_match_cols:
    raise ValueError(f"Leakage detected: feature(s) nearly identical to target -> {exact_match_cols}")

# Dtypes
cat_cols = [
    c for c in X_train.columns
    if X_train[c].dtype == "object"
    or str(X_train[c].dtype).startswith("category")
    or str(X_train[c].dtype) == "bool"
]
num_cols = [c for c in X_train.columns if c not in cat_cols]

print("Target:", target_col)
print("Rows dropped (train/test) due to lag NaNs:", dropped_train, dropped_test)
print("Train/Test shapes after lag prep:", train_df.shape, test_df.shape)
if time_col in train_df.columns and time_col in test_df.columns:
    print("Train max time:", train_df[time_col].max(), "| Test min time:", test_df[time_col].min())
print("Features:", len(feature_cols), "| Numeric:", len(num_cols), "| Categorical:", len(cat_cols))
print("Feature list:", feature_cols)

X_train.head(3)




In [ ]:
# Strict time-aware validation split (no timestamp overlap between fit/valid)
if time_col is not None and time_col in train_df.columns:
    time_series = pd.to_datetime(train_df[time_col], errors="coerce")
    unique_times = np.sort(time_series.dropna().unique())

    if len(unique_times) < 2:
        raise ValueError("Not enough unique timestamps to create validation split.")

    valid_time_count = max(1, int(len(unique_times) * 0.2))
    valid_start_time = unique_times[-valid_time_count]

    fit_mask = time_series < valid_start_time
    valid_mask = time_series >= valid_start_time

    X_fit = X_train.loc[fit_mask].copy()
    y_fit = y_train.loc[fit_mask].copy()

    X_valid = X_train.loc[valid_mask].copy()
    y_valid = y_train.loc[valid_mask].copy()

    print("Validation starts at:", pd.to_datetime(valid_start_time))
    print("Fit max time:", time_series[fit_mask].max(), "| Valid min time:", time_series[valid_mask].min())
else:
    split_idx = int(len(X_train) * 0.8)
    X_fit = X_train.iloc[:split_idx].copy()
    y_fit = y_train.iloc[:split_idx].copy()

    X_valid = X_train.iloc[split_idx:].copy()
    y_valid = y_train.iloc[split_idx:].copy()

print("Fit split:", X_fit.shape, "| Validation split:", X_valid.shape)



In [ ]:
# NaN-safe preprocessing pipelines
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

transformers = []
if cat_cols:
    transformers.append(("cat", categorical_pipeline, cat_cols))
if num_cols:
    transformers.append(("num", numeric_pipeline, num_cols))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
)


def make_model_from_trial(trial):
    options = ["LR", "RIDGE", "RF", "GBR"]
    if HAS_XGB:
        options.append("XGBR")

    model_name = trial.suggest_categorical("model_name", options)

    if model_name == "LR":
        model = LinearRegression()

    elif model_name == "RIDGE":
        alpha = trial.suggest_float("ridge_alpha", 0.1, 200.0, log=True)
        model = Ridge(alpha=alpha, random_state=RANDOM_STATE)

    elif model_name == "RF":
        n_estimators = trial.suggest_int("rf_n_estimators", 100, 400, step=50)
        max_depth = trial.suggest_int("rf_max_depth", 5, 24)
        min_samples_leaf = trial.suggest_int("rf_min_samples_leaf", 1, 8)
        model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    elif model_name == "GBR":
        n_estimators = trial.suggest_int("gbr_n_estimators", 80, 400, step=40)
        learning_rate = trial.suggest_float("gbr_learning_rate", 0.01, 0.2, log=True)
        max_depth = trial.suggest_int("gbr_max_depth", 2, 8)
        subsample = trial.suggest_float("gbr_subsample", 0.6, 1.0)
        model = GradientBoostingRegressor(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            subsample=subsample,
            random_state=RANDOM_STATE,
        )

    else:  # XGBR
        n_estimators = trial.suggest_int("xgb_n_estimators", 120, 500, step=40)
        learning_rate = trial.suggest_float("xgb_learning_rate", 0.01, 0.2, log=True)
        max_depth = trial.suggest_int("xgb_max_depth", 3, 10)
        subsample = trial.suggest_float("xgb_subsample", 0.6, 1.0)
        colsample_bytree = trial.suggest_float("xgb_colsample", 0.6, 1.0)
        model = XGBRegressor(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    return model_name, model


def objective(trial):
    model_name, model = make_model_from_trial(trial)

    pipeline = Pipeline([
        ("prep", preprocessor),
        ("model", model),
    ])

    # Optional subsampling for faster Optuna iterations on large data.
    sample_size = min(50_000, len(X_fit))
    if sample_size < len(X_fit):
        rng = np.random.default_rng(RANDOM_STATE + trial.number)
        idx = rng.choice(len(X_fit), size=sample_size, replace=False)
        X_fit_sample = X_fit.iloc[idx]
        y_fit_sample = y_fit.iloc[idx]
    else:
        X_fit_sample = X_fit
        y_fit_sample = y_fit

    if USE_MLFLOW and HAS_MLFLOW:
        mlflow.start_run(nested=True)
        mlflow.log_param("model_name", model_name)

    pipeline.fit(X_fit_sample, y_fit_sample)
    y_pred_valid = pipeline.predict(X_valid)

    metrics = evaluate_metrics(y_valid, y_pred_valid)

    # Leakage-sanity guard on metric scale.
    if metrics["MAPE"] < 1e-3:
        raise ValueError(f"Leakage suspected: unrealistically low validation MAPE={metrics['MAPE']:.3e}")

    if USE_MLFLOW and HAS_MLFLOW:
        for k, v in metrics.items():
            mlflow.log_metric(f"valid_{k}", v)
        mlflow.log_params(model.get_params())
        mlflow.end_run()

    return metrics["MAPE"]




In [ ]:
# Run Optuna model selection
n_trials = 80

study = optuna.create_study(
    study_name="model_selection",
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10),
)

if USE_MLFLOW and HAS_MLFLOW:
    with mlflow.start_run(run_name="model_selection_optuna"):
        study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True, catch=(ValueError,))
        mlflow.log_params(study.best_params)
        mlflow.log_metric("best_valid_MAPE", study.best_value)
else:
    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=True, catch=(ValueError,))

print("Best validation MAPE:", round(study.best_value, 6))
print("Best params:")
study.best_params



In [ ]:
# Trials leaderboard
trials_df = study.trials_dataframe()
leaderboard_cols = [
    "number",
    "value",
    "params_model_name",
    "state",
]

leaderboard = trials_df[[c for c in leaderboard_cols if c in trials_df.columns]].copy()
leaderboard = leaderboard.rename(columns={"value": "valid_MAPE"}).sort_values("valid_MAPE").reset_index(drop=True)
leaderboard.head(10)


In [ ]:
# Train final model on full train set and evaluate on test set
class DummyTrial:
    def __init__(self, params):
        self.params = params

    def suggest_categorical(self, name, choices):
        return self.params[name]

    def suggest_int(self, name, low, high, step=1):
        return int(self.params[name])

    def suggest_float(self, name, low, high, log=False):
        return float(self.params[name])


best_trial = DummyTrial(study.best_params)
best_model_name, best_model = make_model_from_trial(best_trial)

best_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", best_model),
])

best_pipeline.fit(X_train, y_train)

y_pred_train = np.clip(best_pipeline.predict(X_train), a_min=0, a_max=None)
y_pred_test = np.clip(best_pipeline.predict(X_test), a_min=0, a_max=None)

train_metrics = evaluate_metrics(y_train, y_pred_train)
test_metrics = evaluate_metrics(y_test, y_pred_test)

metrics_summary = pd.DataFrame([
    {"split": "train", **train_metrics},
    {"split": "test", **test_metrics},
])

print("Final model:", best_model_name)
metrics_summary


In [ ]:
# Build prediction output table
predictions = pd.DataFrame({
    "actual_demand": y_test.values,
    "predicted_demand": y_pred_test,
}, index=y_test.index).reset_index(drop=True)

if time_col is not None and time_col in context_test.columns:
    predictions["pickup_slot"] = pd.to_datetime(context_test[time_col].values)
else:
    predictions["pickup_slot"] = pd.RangeIndex(start=0, stop=len(predictions), step=1)

if "region" in context_test.columns:
    predictions["region"] = pd.to_numeric(context_test["region"].values, errors="coerce").astype("Int64")
else:
    predictions["region"] = pd.Series([pd.NA] * len(predictions), dtype="Int64")

# Bring context features for business logic
for col in ["rolling_mean", "rolling_std", "avg_fare_region_slot", "avg_pickups", "avg_pickups_ewm"]:
    if col in context_test.columns:
        predictions[col] = pd.to_numeric(context_test[col].values, errors="coerce")

predictions = predictions.sort_values(["region", "pickup_slot"], na_position="last").reset_index(drop=True)
predictions.head()


In [ ]:
# Business intelligence layer from model predictions

# 1) Surge detection
if "rolling_mean" not in predictions.columns:
    predictions["rolling_mean"] = predictions.groupby("region")["actual_demand"].transform("mean")
predictions["rolling_mean"] = predictions["rolling_mean"].fillna(predictions["rolling_mean"].median())

# Notebook-5 requested rule: threshold = rolling_mean * 1.3
predictions["surge_threshold"] = predictions["rolling_mean"] * 1.3
predictions["surge_flag"] = predictions["predicted_demand"] > predictions["surge_threshold"]

surge_ratio = predictions["predicted_demand"] / (predictions["surge_threshold"] + 1e-6)
predictions["surge_level"] = "none"
predictions.loc[predictions["surge_flag"] & (surge_ratio <= 1.10), "surge_level"] = "low"
predictions.loc[predictions["surge_flag"] & (surge_ratio > 1.10) & (surge_ratio <= 1.25), "surge_level"] = "medium"
predictions.loc[predictions["surge_flag"] & (surge_ratio > 1.25), "surge_level"] = "high"

# 2) Risk / stability score
if "rolling_std" not in predictions.columns:
    predictions["rolling_std"] = predictions.groupby("region")["actual_demand"].transform("std")
predictions["rolling_std"] = predictions["rolling_std"].fillna(0)

predictions["risk_score"] = predictions["rolling_std"] / (predictions["rolling_mean"] + 1e-6)
predictions["risk_band"] = pd.cut(
    predictions["risk_score"],
    bins=[-np.inf, 0.35, 0.75, np.inf],
    labels=["Stable", "Moderate", "Volatile"],
).astype(str)

# 3) Revenue estimation
if "avg_fare_region_slot" not in predictions.columns:
    fare_baseline = train_df["avg_fare_region_slot"].median() if "avg_fare_region_slot" in train_df.columns else np.nan
    predictions["avg_fare_region_slot"] = fare_baseline

predictions["avg_fare_region_slot"] = predictions["avg_fare_region_slot"].fillna(predictions["avg_fare_region_slot"].median())
predictions["expected_revenue"] = predictions["predicted_demand"] * predictions["avg_fare_region_slot"]

# 4) Demand pressure
if "avg_pickups" in predictions.columns:
    denom = predictions["avg_pickups"]
elif "avg_pickups_ewm" in predictions.columns:
    denom = predictions["avg_pickups_ewm"]
else:
    denom = predictions.groupby("region")["actual_demand"].transform("mean")

predictions["demand_pressure"] = predictions["predicted_demand"] / (pd.to_numeric(denom, errors="coerce") + 1e-6)


In [ ]:
# Driver relocation recommendation using neighbor regions
neighbors_path = data_interim / "region_neighbors.csv"

if neighbors_path.exists() and predictions["region"].notna().any():
    neighbors = pd.read_csv(neighbors_path)
    neighbors = neighbors.rename(columns={"region_id": "region", "neighbor_region_id": "target_region"})

    base = predictions[["pickup_slot", "region", "predicted_demand"]].copy()
    candidate_moves = base.merge(neighbors[["region", "target_region", "distance_km"]], on="region", how="left")

    target_demand = base.rename(
        columns={
            "region": "target_region",
            "predicted_demand": "target_predicted_demand",
        }
    )
    candidate_moves = candidate_moves.merge(target_demand, on=["pickup_slot", "target_region"], how="left")

    candidate_moves["target_predicted_demand"] = candidate_moves["target_predicted_demand"].fillna(0)
    candidate_moves["demand_gain"] = (
        candidate_moves["target_predicted_demand"] - candidate_moves["predicted_demand"]
    ).clip(lower=0)
    candidate_moves["relocation_score"] = candidate_moves["demand_gain"] / (candidate_moves["distance_km"] + 1e-3)

    top_moves = (
        candidate_moves.sort_values(["pickup_slot", "region", "relocation_score"], ascending=[True, True, False])
        .groupby(["pickup_slot", "region"], as_index=False)
        .head(3)
        .copy()
    )
    top_moves["rank"] = top_moves.groupby(["pickup_slot", "region"]).cumcount() + 1

    best_moves = top_moves[top_moves["rank"] == 1].copy()
    min_gain = 1.0
    low_gain = best_moves["demand_gain"] < min_gain
    best_moves.loc[low_gain, ["target_region", "distance_km", "demand_gain", "relocation_score"]] = [np.nan, np.nan, 0.0, 0.0]

    best_moves = best_moves.rename(
        columns={
            "target_region": "recommended_next_zone",
            "distance_km": "recommended_distance_km",
            "demand_gain": "expected_demand_gain",
        }
    )

    predictions = predictions.merge(
        best_moves[["pickup_slot", "region", "recommended_next_zone", "recommended_distance_km", "expected_demand_gain", "relocation_score"]],
        on=["pickup_slot", "region"],
        how="left",
    )
else:
    top_moves = pd.DataFrame()
    predictions["recommended_next_zone"] = pd.Series([pd.NA] * len(predictions), dtype="Int64")
    predictions["recommended_distance_km"] = np.nan
    predictions["expected_demand_gain"] = 0.0
    predictions["relocation_score"] = 0.0


In [ ]:
# Best-time recommendation by region from model predictions
if np.issubdtype(predictions["pickup_slot"].dtype, np.datetime64):
    predictions["pickup_day_of_week"] = predictions["pickup_slot"].dt.dayofweek
    predictions["pickup_hour"] = predictions["pickup_slot"].dt.hour

    slot_profile = (
        predictions.groupby(["region", "pickup_day_of_week", "pickup_hour"], dropna=False, as_index=False)["predicted_demand"]
        .mean()
        .rename(columns={"predicted_demand": "avg_predicted_demand"})
    )

    best_time_recommendations = (
        slot_profile.sort_values(["region", "avg_predicted_demand"], ascending=[True, False])
        .groupby("region", as_index=False)
        .head(3)
        .copy()
    )
    best_time_recommendations["rank"] = best_time_recommendations.groupby("region").cumcount() + 1

    day_names = {
        0: "Monday", 1: "Tuesday", 2: "Wednesday", 3: "Thursday",
        4: "Friday", 5: "Saturday", 6: "Sunday",
    }
    best_time_recommendations["day_name"] = best_time_recommendations["pickup_day_of_week"].map(day_names)
    best_time_recommendations["best_time_window"] = best_time_recommendations["pickup_hour"].astype(int).map(
        lambda h: f"{h:02d}:00-{(h + 1) % 24:02d}:00"
    )

    top1_time = (
        best_time_recommendations[best_time_recommendations["rank"] == 1][["region", "best_time_window"]]
        .drop_duplicates(subset=["region"])
    )
    predictions = predictions.merge(top1_time, on="region", how="left")
else:
    slot_profile = pd.DataFrame()
    best_time_recommendations = pd.DataFrame()
    predictions["best_time_window"] = np.nan


In [ ]:
# Save artifacts and outputs
model_path = models_dir / "best_model_selection_pipeline.joblib"
leaderboard_path = reports_dir / "model_selection_leaderboard.csv"
metrics_path = reports_dir / "model_metrics_summary.csv"

pred_path = data_interim / "model_test_predictions.csv"
business_output_path = data_interim / "model_business_output.csv"
relocation_path = data_interim / "model_relocation_candidates.csv"
best_time_path = data_interim / "model_best_time_recommendations.csv"

joblib.dump(best_pipeline, model_path)

leaderboard.to_csv(leaderboard_path, index=False)
metrics_summary.to_csv(metrics_path, index=False)

predictions.to_csv(pred_path, index=False)

ui_cols = [
    "pickup_slot",
    "region",
    "actual_demand",
    "predicted_demand",
    "surge_flag",
    "surge_level",
    "risk_score",
    "risk_band",
    "expected_revenue",
    "demand_pressure",
    "recommended_next_zone",
    "recommended_distance_km",
    "expected_demand_gain",
    "relocation_score",
    "best_time_window",
]
predictions[[c for c in ui_cols if c in predictions.columns]].to_csv(business_output_path, index=False)

if not top_moves.empty:
    top_moves.to_csv(relocation_path, index=False)
else:
    pd.DataFrame().to_csv(relocation_path, index=False)

if not best_time_recommendations.empty:
    best_time_recommendations.to_csv(best_time_path, index=False)
else:
    pd.DataFrame().to_csv(best_time_path, index=False)

print("Saved artifacts:")
for p in [
    model_path,
    leaderboard_path,
    metrics_path,
    pred_path,
    business_output_path,
    relocation_path,
    best_time_path,
]:
    print("-", p.relative_to(project_root))


In [ ]:
# Quick checks
print("Best model:", best_model_name)
display(metrics_summary)
display(predictions.head(10))


In [ ]:
# Simple plot: actual vs predicted demand (test)
plot_df = predictions.copy()
if np.issubdtype(plot_df["pickup_slot"].dtype, np.datetime64):
    plot_df = plot_df.sort_values("pickup_slot")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(plot_df["actual_demand"].values[:1000], label="Actual", alpha=0.8)
ax.plot(plot_df["predicted_demand"].values[:1000], label="Predicted", alpha=0.8)
ax.set_title("Actual vs Predicted Demand (first 1000 test rows)")
ax.set_xlabel("Row")
ax.set_ylabel("Demand")
ax.legend()
plt.tight_layout()
plt.show()


## Notes
- This notebook keeps old model-selection strengths (Optuna + model comparison) and adds business-ready outputs.
- Notebook 4 features are consumed when available (`historical_features.csv`), otherwise fallback is `processed/train.csv` and `processed/test.csv`.
- `model_business_output.csv` is a UI-friendly output with predicted demand + surge/risk/revenue/relocation/best-time fields.
- Lag features are created before model selection (`lag_1`, `lag_2`, `lag_3`, `lag_6`, `lag_12`, `lag_96`, plus shifted rolling lag stats).
